# Direct localization evaluation on updated fingerprints

This evaluates whether updated **RSS fingerprints** help localize new scans. It complements `evaluation_in_stages.ipynb`, which evaluates GNN embeddings.

For each representation, a fresh localization MLP is trained on labeled reference fingerprints only, then evaluated on the same held-out update scans. Update coordinates are used only to score predictions.

| Stage | Reference fingerprints used for training | Update scans used for evaluation |
|---|---|---|
| `original` | Before the batch transition | Measured update RSS, aligned to the original AP vocabulary |
| `round1` | First updated reference fingerprints | The same measured update RSS, aligned to the round-1 AP vocabulary |
| `round2` | Second updated reference fingerprints | The same measured update RSS, aligned to the round-2 AP vocabulary |

Each stage fits its own probe using the same architecture and training settings. The comparison measures whether each reference fingerprint space supports localization of the new scans; it does not reuse one model's weights across different AP vocabularies.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from gufu_direct_evaluation import evaluate_direct_run

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'gufu.py').is_file():
    raise RuntimeError('Run this notebook with the GUFU repository root as the working directory.')

OUTPUT_ROOTS = {
    'GF_from1F_12': PROJECT_ROOT / 'output_from1F_12',
    'GF_fromGF_12': PROJECT_ROOT / 'output_fromGF_12',
}
DEVICE_OVERRIDE = None  # None uses the device recorded in each run manifest.

## Run both saved experiments

The next cell reads the completed snapshots under `output_from1F_12` and `output_fromGF_12`. It does not rerun GNN training, AE training, or the streaming pipeline; it only fits the evaluation probes and scores the saved fingerprints against the held-out update locations.

In [2]:
# Evaluation-only pass over the existing saved outputs; no pipeline retraining.
summary_rows = []
per_scan_frames = []
for run_name, output_root in OUTPUT_ROOTS.items():
    if not output_root.is_dir():
        raise FileNotFoundError(f"Saved output directory not found: {output_root}")
    print(f"Evaluating direct fingerprints for {run_name} from {output_root}...")
    run_summary, run_per_scan = evaluate_direct_run(
        run_name, output_root, device_override=DEVICE_OVERRIDE
    )
    summary_rows.extend(run_summary)
    per_scan_frames.append(run_per_scan)

summary = pd.DataFrame(summary_rows)
per_scan = pd.concat(per_scan_frames, ignore_index=True)
stage_order = {'original': 0, 'round1': 1, 'round2': 2}
summary['_stage_order'] = summary['stage'].map(stage_order)
summary = summary.sort_values(['run', '_stage_order']).drop(columns='_stage_order').reset_index(drop=True)
display(summary)

Evaluating direct fingerprints for GF_from1F_12 from /home/oem/GUFU/output_from1F_12...
Evaluating direct fingerprints for GF_fromGF_12 from /home/oem/GUFU/output_fromGF_12...


,run,stage,held_out_scans,labeled_reference_scans,AP_count,mean_update_observed_APs,mean_update_AP_coverage_percent,2D_RMSE,RMSE_delta_vs_original,RMSE_change_percent_vs_original,mean_radial_error,median_radial_error,p90_radial_error,scans_improved_vs_original,scans_worsened_vs_original
0,GF_from1F_12,original,419,258,317,40.818616,12.876535,40.225253,0.000000,0.000000,30.154611,21.136283,75.018043,0,0
1,GF_from1F_12,round1,419,258,319,41.047733,12.867628,60.117812,19.892559,49.452914,48.249755,36.393157,111.459253,55,364
2,GF_from1F_12,round2,419,258,319,41.047733,12.867628,65.824810,25.599557,63.640513,53.472173,39.894180,119.422094,50,369
3,GF_fromGF_12,original,1733,2078,1022,36.507790,3.572191,34.276166,0.000000,0.000000,25.156239,20.671075,54.128758,0,0
4,GF_fromGF_12,round1,1733,2078,1041,38.828621,3.729935,60.172162,25.895997,75.551032,51.976222,48.816333,93.720178,203,1530
5,GF_fromGF_12,round2,1733,2078,1041,38.828621,3.729935,57.799113,23.522948,68.627711,50.642359,49.663594,89.531464,246,1487


## Per-scan changes

Negative deltas mean the updated fingerprints reduced that scan's localization error relative to original fingerprints. Positive deltas mean they increased it.

In [3]:
worst_round1 = (
    per_scan.sort_values('round1_minus_original_error', ascending=False)
    .groupby('run', as_index=False).head(10)
)
worst_round2 = (
    per_scan.sort_values('round2_minus_original_error', ascending=False)
    .groupby('run', as_index=False).head(10)
)

print('Largest round-1 regressions vs original:')
display(worst_round1[['run', 'scan_id', 'original_error', 'round1_error', 'round1_minus_original_error']])
print('Largest round-2 regressions vs original:')
display(worst_round2[['run', 'scan_id', 'original_error', 'round2_error', 'round2_minus_original_error']])

Largest round-1 regressions vs original:


,run,scan_id,original_error,round1_error,round1_minus_original_error
450,GF_fromGF_12,scan:ae7128cb37ca08b07c159c1a9707be0e00fdd8037...,10.014025,145.626641,135.612615
878,GF_fromGF_12,scan:3f1b7af201b2c35f3a1b8191a8c6469087f986e22...,30.737898,164.443069,133.705171
440,GF_fromGF_12,scan:59e1d6220b6fad97a4668fe4cc6118eb0e444aef3...,5.589429,134.472358,128.882929
881,GF_fromGF_12,scan:fff0c504c597263f885cb07163be01dc544d92671...,34.519085,157.207433,122.688348
502,GF_fromGF_12,scan:b414f985ad60a5b46aefb21528558e457a41fce1e...,1.760235,123.367980,121.607745
859,GF_fromGF_12,scan:b63134d82a643ec11ae7ff72a0f25ef944d5d1e77...,6.921876,128.285020,121.363144
501,GF_fromGF_12,scan:bbf91cc6c073281270df93b7fa68901b3b3c48067...,26.104332,147.076693,120.972360
455,GF_fromGF_12,scan:dfc4067c8af5a01a9f3c12d93cdc08228b1addd1f...,24.180012,143.932086,119.752074
503,GF_fromGF_12,scan:3d3d3f0446e3f7620caa50c3bbf48f3ab14fe94e8...,7.258937,124.627549,117.368612
886,GF_fromGF_12,scan:dfeeed6674ec4b26518989f2f60837f5789be6a05...,45.127843,159.561058,114.433214


Largest round-2 regressions vs original:


,run,scan_id,original_error,round2_error,round2_minus_original_error
450,GF_fromGF_12,scan:ae7128cb37ca08b07c159c1a9707be0e00fdd8037...,10.014025,106.960120,96.946094
792,GF_fromGF_12,scan:4bfa581c9c0ad338a2a9a5621dc0b2a2b465afa8f...,6.122018,100.933986,94.811968
878,GF_fromGF_12,scan:3f1b7af201b2c35f3a1b8191a8c6469087f986e22...,30.737898,121.162450,90.424551
440,GF_fromGF_12,scan:59e1d6220b6fad97a4668fe4cc6118eb0e444aef3...,5.589429,93.886392,88.296963
859,GF_fromGF_12,scan:b63134d82a643ec11ae7ff72a0f25ef944d5d1e77...,6.921876,95.184118,88.262242
957,GF_fromGF_12,scan:e3dc2b37f062c385dd2129f80aa0569253e8045a4...,36.574548,124.528992,87.954444
966,GF_fromGF_12,scan:7643c2b7c1f1e7f2472d3e619a54c5e1ce38d2aa7...,14.005973,100.929182,86.923209
914,GF_fromGF_12,scan:8aae6ca9795a194f3566af9718c2162294820fabb...,15.413421,102.300569,86.887148
979,GF_fromGF_12,scan:bb62d378dc66b3295355856aed9565fff41b3d1a1...,16.960544,103.694106,86.733562
89,GF_from1F_12,scan:cbc11dcb46af5e8a6fd01e4e441eb649870a9545a...,15.464751,101.685601,86.220850


## Reading the result

The primary comparison is `2D_RMSE` and `RMSE_delta_vs_original` on held-out update scans. This is an end-to-end localization check on RSS fingerprints, unlike the stage evaluation on GNN embeddings. A lower error means a reference-trained probe transfers better to the new scans. It does not show that every inferred AP value is physically accurate, and it should be read alongside scan coverage and the per-scan errors above.